In [1]:
# =====================================================================
#  Cell 1 — Imports & Device Setup
# =====================================================================
import json
import re
import torch
import config  # contains GEMINI_API_KEY

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
)

import google.generativeai as genai
from google.generativeai import types

print("✅ Imports loaded.")

# Detect if GPU is available (faster inference)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

c:\Users\LENOVO\Downloads\NLP_Proj\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\LENOVO\Downloads\NLP_Proj\.venv\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ Imports loaded.
Using device: cuda


In [2]:
# =====================================================================
#  Cell 2 — Load CAMeLBERT Classification Model
#  This model predicts the *topic* of the news article.
# =====================================================================

CAMEL_MODEL_DIR = r"C:\Users\LENOVO\Downloads\NLP_Proj\NLP_PROJ\Models\CAMeLBERT_classifier"

camel_tokenizer = AutoTokenizer.from_pretrained(CAMEL_MODEL_DIR)
camel_model = AutoModelForSequenceClassification.from_pretrained(CAMEL_MODEL_DIR)

camel_model.to(device)
camel_model.eval()

print("✓ CAMeLBERT classifier loaded successfully.")
print("Labels map:", camel_model.config.id2label)

# English → Arabic topic names mapping
EN2AR_TOPIC = {
    "Sports": "الرياضة",
    "Politics": "السياسة",
    "Medical": "الطب",
    "Tech": "التقنية",
    "Culture": "الثقافة",
    "Finance": "الاقتصاد",
    "Religion": "الدين",
}

✓ CAMeLBERT classifier loaded successfully.
Labels map: {0: 'Culture', 1: 'Finance', 2: 'Medical', 3: 'Politics', 4: 'Religion', 5: 'Sports', 6: 'Tech'}


In [3]:
# =====================================================================
#  Cell 3 — Load AraT5 Summarization Model
#  This model generates a short summary for each news article.
# =====================================================================

ARAT5_MODEL_DIR = r"C:\Users\LENOVO\Downloads\NLP_Proj\NLP_PROJ\Models\AraT5_Summarization"

t5_tokenizer = AutoTokenizer.from_pretrained(ARAT5_MODEL_DIR)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(ARAT5_MODEL_DIR)

t5_model.to(device)
t5_model.eval()

print("✓ AraT5 summarizer loaded successfully.")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


✓ AraT5 summarizer loaded successfully.


In [4]:
# =====================================================================
#  Cell 4 — Gemini Configuration (5W Extraction)
# =====================================================================

genai.configure(api_key=config.GEMINI_API_KEY)

# System instruction for Gemini (JSON-only behaviour)
sys_instruction = """
أنت خبير استخراج بيانات.
مهمتك: استخراج (who, what, when, where, source) من النصوص العربية.
يجب أن تكون المخرجات حصراً بصيغة JSON صالح (Valid JSON).
"""

generation_config = types.GenerationConfig(
    temperature=0.0,
    top_p=0.9,
    max_output_tokens=1024,
    response_mime_type="application/json",
)

gemini_model = genai.GenerativeModel(
    model_name="models/gemini-2.5-flash",
    generation_config=generation_config,
    system_instruction=sys_instruction,
)

print("✓ Gemini API connected successfully (JSON Mode Enabled).")

✓ Gemini API connected successfully (JSON Mode Enabled).


In [5]:
# =====================================================================
#  Cell 5 — Helper Functions: Classification (CAMeLBERT)
# =====================================================================

def classify_with_camelbert(text: str, max_length: int = 512):
    """
    Run the CAMeLBERT classifier on the given text and return:
    - topic_id
    - topic_name_en
    - topic_name (Arabic label)
    """
    inputs = camel_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = camel_model(**inputs)
        pred_id = int(outputs.logits.argmax(dim=-1).item())

    en_label = camel_model.config.id2label[pred_id]
    ar_label = EN2AR_TOPIC.get(en_label, en_label)

    return {
        "topic_id": pred_id,
        "topic_name_en": en_label,
        "topic_name": ar_label,
    }

In [6]:
# =====================================================================
#  Cell 6 — Helper Functions: Summarization (AraT5)
# =====================================================================

def clean_repeated_tail(sentence: str) -> str:
    """
    Removes repeated tail patterns like:
    '... في المملكة العربية السعودية في المملكة العربية السعودية'
    """
    words = sentence.split()
    for k in range(3, len(words) // 2 + 1):
        tail = words[-k:]
        prev = words[-2 * k : -k]
        if tail == prev and tail:
            words = words[:-k]
            break
    return " ".join(words)


def smart_deduplicate(points, min_len: int = 10):
    """
    Removes very short / duplicate / almost-duplicate points.
    Keeps up to 3 clean points.
    """
    cleaned = []
    for p in points:
        p_stripped = p.strip()
        if len(p_stripped) < min_len:
            continue
        if any(p_stripped == q for q in cleaned):
            continue
        if any(p_stripped in q or q in p_stripped for q in cleaned):
            continue
        cleaned.append(p_stripped)
    return cleaned[:3]


def parse_points(summary_text: str) -> str:
    """
    Converts AraT5 raw summary into:
      1. ...
      2. ...
      3. ...
    if possible.
    """
    text = summary_text.replace("\n", " ")
    parts = re.split(r"\s*[1-3]\s*[\.\-،:]\s*", text)
    raw_points = [p for p in parts[1:] if p.strip()]
    raw_points = [clean_repeated_tail(p) for p in raw_points]
    points = smart_deduplicate(raw_points)
    numbered = [f"{i+1}. {p}" for i, p in enumerate(points)]
    return "\n".join(numbered)


def summarize_with_arat5(
    text: str, max_input_length: int = 750, max_target_length: int = 150
) -> str:
    """
    Uses the fine-tuned AraT5 model to summarize the input article.
    Returns a CLEAN 3-point summary (1./2./3.) as text.
    """
    enc = t5_tokenizer(
        text,
        max_length=max_input_length,
        truncation=True,
        padding=False,
        return_tensors="pt",
    )

    input_ids = enc.input_ids.to(device)

    with torch.no_grad():
        outputs = t5_model.generate(
            input_ids,
            max_length=max_target_length,
            num_beams=4,
            no_repeat_ngram_size=4,
            repetition_penalty=1.2,
            length_penalty=0.8,
            early_stopping=True,
        )

    raw_summary = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    cleaned_summary = parse_points(raw_summary)
    return cleaned_summary

In [7]:
# =====================================================================
#  Cell 7 — Helper Functions: 5W Extraction (Gemini)
# =====================================================================

FIVEW_PROMPT_TEMPLATE = """
مهمتك استخراج المعلومات من النص التالي بصيغة JSON.

الحقول المطلوبة:
1. who: الشخصيات أو الجهات الفاعلة (الأسماء الصريحة).
2. what: الحدث الرئيسي (ملخص في جملة واحدة).
3. when: الزمان.
4. where: المكان.
5. source: المصدر.

قواعد:
- إذا لم تتوفر المعلومة اكتب "غير مذكور".
- المخرجات JSON فقط.

النص:
{text}
"""

def extract_5w_with_gemini(text: str):
    """
    Uses Gemini (JSON mode) to extract:
      who, what, when, where, source
    Returns: Python dict or None on failure.
    """
    prompt = FIVEW_PROMPT_TEMPLATE.replace("{text}", text)

    try:
        response = gemini_model.generate_content(prompt)

        if not response.candidates:
            print("⚠️ Gemini لم يرجع أي candidates.")
            return None

        # response.text should already be a JSON string (because of JSON mime type)
        raw_text = response.text.strip()

        # remove markdown fences if any
        raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        data = json.loads(raw_text)
        return data

    except Exception as e:
        print("❌ Error in extract_5w_with_gemini:", e)
        return None

In [8]:
# =====================================================================
#  Cell 8 — Full Unified Pipeline
# =====================================================================

def process_article_full(text: str):
    """
    Run the full pipeline on a given article:
      1) Topic classification (CAMeLBERT)
      2) Summarization (AraT5)
      3) 5W extraction (Gemini)
    """
    # 1. Classification
    topic = classify_with_camelbert(text)

    # 2. Summarization
    summary = summarize_with_arat5(text)

    # 3. 5W Extraction
    five_w_data = extract_5w_with_gemini(text)

    # Fallback if extraction failed
    if not five_w_data:
        five_w_data = {
            "who": "غير مذكور",
            "what": "غير مذكور",
            "when": "غير مذكور",
            "where": "غير مذكور",
            "source": "غير مذكور",
        }

    return {
        "topic": topic,
        "summary": summary,
        "five_w": five_w_data,
    }

print("✓ Unified News AI Pipeline is ready (Production Ready).")

✓ Unified News AI Pipeline is ready (Production Ready).


In [9]:
# =====================================================================
#  Cell 9 — Pretty Arabic Output Formatter
# =====================================================================

def format_output_arabic(result: dict):
    """
    Nicely print the pipeline result in Arabic with colors.
    """
    class Colors:
        HEADER = '\033[95m'
        BLUE   = '\033[94m'
        CYAN   = '\033[96m'
        GREEN  = '\033[92m'
        YELLOW = '\033[93m'
        BOLD   = '\033[1m'
        END    = '\033[0m'
        GREY   = '\033[90m'

    topic = result.get("topic", {})
    summary = result.get("summary", "")
    fivew = result.get("five_w") or {}

    def clean(val):
        if val is None or str(val).strip() in ["null", "None", ""]:
            return f"{Colors.GREY}غير مذكور{Colors.END}"
        if isinstance(val, list):
            val = "، ".join(map(str, val))
        return str(val).strip()

    t_name = topic.get("topic_name", "غير معروف")

    print("\n")
    print(f"{Colors.BOLD}{Colors.HEADER}{'━'*70}{Colors.END}")
    print(
        f"{Colors.BOLD}{Colors.HEADER}📌 تقرير تحليل الخبر{Colors.END}"
        f"  |  {Colors.YELLOW}التصنيف: {t_name}{Colors.END}"
    )
    print(f"{Colors.BOLD}{Colors.HEADER}{'━'*70}{Colors.END}")

    # Summary
    print(f"\n{Colors.BOLD}{Colors.CYAN}📝 ملخص الأحداث:{Colors.END}")
    if summary:
        summary_lines = [l.strip() for l in summary.split("\n") if l.strip()]
        for line in summary_lines:
            print(f"  {line}")   # no extra numbering here, AraT5 already adds 1./2./3.
    else:
        print(f"  {Colors.GREY}لا يوجد ملخص متاح.{Colors.END}")

    print(f"\n{Colors.GREY}{'─'*56}{Colors.END}")

    # 5W
    print(f"{Colors.BOLD}{Colors.CYAN}🔍 استخراج المعلومات (5W):{Colors.END}\n")

    items = [
        ("👤 من (Who)",        fivew.get("who")),
        ("⚡ ماذا (What)",     fivew.get("what")),
        ("📅 متى (When)",      fivew.get("when")),
        ("📍 أين (Where)",     fivew.get("where")),
        ("📰 المصدر (Source)", fivew.get("source")),
    ]

    for label, value in items:
        val_cleaned = clean(value)
        print(f"{Colors.BLUE}{label:<18}{Colors.END} : {val_cleaned}")

    print(f"\n{Colors.GREY}{'─'*56}{Colors.END}")

In [ ]:
# =====================================================================
#  Cell 10 — Test the Full Pipeline on Sample Articles
# =====================================================================

article1 = """أعلنت وزارة التعليم في المملكة العربية السعودية صباح اليوم الثلاثاء عن بدء تنفيذ مشروع جديد
يهدف إلى تطوير المناهج الرقمية في المدارس الحكومية، وذلك بالتعاون مع شركة "تقنية التعليم الحديثة" المختصة في حلول التعليم الذكي.

وذكرت الوزارة في بيان رسمي نشرته على موقعها الإلكتروني أن المشروع سيبدأ تطبيقه خلال الفصل الدراسي القادم،
وسيستهدف أكثر من 1500 مدرسة في مختلف مناطق المملكة، بهدف رفع جودة المحتوى التعليمي
وتحسين تجربة الطلاب من خلال منصات تفاعلية وتقنيات الواقع المعزز.

وأوضح البيان أن المشروع يتضمن تدريب أكثر من 8000 معلم ومعلمة على استخدام الأنظمة الرقمية الجديدة،
إضافة إلى توفير بنية تحتية متقدمة تشمل أجهزة لوحية للطلاب وخوادم مدرسية لحفظ المحتوى والواجبات.

وأكدت وزارة التعليم أن المبادرة تأتي ضمن مستهدفات رؤية السعودية 2030 الهادفة إلى تطوير قطاع التعليم،
وأن اختيار شركة "تقنية التعليم الحديثة" جاء بعد دراسة عدد من العروض التقنية التي قُدِّمت للوزارة.

المصدر: موقع وزارة التعليم السعودي."""

result1 = process_article_full(article1)
format_output_arabic(result1)




━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📌 تقرير تحليل الخبر  |  التصنيف: التقنية
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📝 ملخص الأحداث:
  1. أعلنت وزارة التعليم السعودية بدء مشروع تطوير المناهج الرقمية في المدارس الحكومية بالتعاون مع شركة "تقنية التعليم الحديثة".
  2. يهدف المشروع إلى رفع جودة المحتوى التعليمي وتحسين تجربة الطلاب من خلال منصات تفاعلية وتقنيات الواقع المعزز.
  3. يأتي المشروع ضمن مستهدفات رؤية السعودية 2030 لتطوير قطاع التعليم.

────────────────────────────────────────────────────────
🔍 استخراج المعلومات (5W):

👤 من (Who)         : وزارة التعليم في المملكة العربية السعودية، شركة "تقنية التعليم الحديثة"
⚡ ماذا (What)      : إعلان عن بدء تنفيذ مشروع لتطوير المناهج الرقمية في المدارس الحكومية
📅 متى (When)       : صباح اليوم الثلاثاء
📍 أين (Where)      : المملكة العربية السعودية، يستهدف أكثر من 1500 مدرسة في مختلف مناطق المملكة
📰 المصدر (Source)  : موقع وزارة التعليم السعودي

─────────────────────────────────

In [11]:

article2 = """أعلنت وزارة النقل والخدمات اللوجستية في المملكة العربية السعودية اليوم الأربعاء عن بدء تنفيذ مشروع جديد لتطوير البنية التحتية للطرق في منطقة عسير، وذلك ضمن جهود تعزيز السلامة المرورية وتحسين جودة شبكة الطرق في المنطقة.

وذكرت الوزارة في بيان لها أن المشروع يشمل توسعة وتحديث أكثر من 120 كيلومترًا من الطرق الحيوية، بالإضافة إلى إنشاء جسور وأنفاق جديدة بهدف تسهيل حركة المركبات والحد من الازدحام في الممرات الجبلية.

كما أكدت الوزارة أن تنفيذ المشروع سيتم بالتعاون مع عدد من الشركات الوطنية المتخصصة في قطاع المقاولات، وأنه من المتوقع الانتهاء منه في الربع الأخير من عام 2026.

وأشارت الوزارة إلى أن هذا المشروع يُعد جزءًا من برنامج تطوير شامل يستهدف رفع كفاءة النقل البري في مختلف مناطق المملكة وتعزيز الاتصال بين المدن الرئيسية.

المصدر: وكالة الأنباء السعودية (واس)"""

result2 = process_article_full(article2)
format_output_arabic(result2)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📌 تقرير تحليل الخبر  |  التصنيف: السياسة
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📝 ملخص الأحداث:
  1. أعلنت وزارة النقل السعودية بدء تنفيذ مشروع جديد لتطوير البنية التحتية للطرق في منطقة عسير.
  2. يشمل المشروع توسعة وتحديث أكثر من 120 كيلومترًا من الطرق الحيوية وإنشاء جسور وأنفاق جديدة.
  3. يهدف المشروع إلى رفع كفاءة النقل البري وتعزيز الاتصال بين المدن الرئيسية.

────────────────────────────────────────────────────────
🔍 استخراج المعلومات (5W):

👤 من (Who)         : وزارة النقل والخدمات اللوجستية في المملكة العربية السعودية، وعدد من الشركات الوطنية المتخصصة في قطاع المقاولات
⚡ ماذا (What)      : بدء تنفيذ مشروع جديد لتطوير البنية التحتية للطرق في منطقة عسير
📅 متى (When)       : اليوم الأربعاء (مع توقع الانتهاء في الربع الأخير من عام 2026)
📍 أين (Where)      : منطقة عسير، المملكة العربية السعودية
📰 المصدر (Source)  : وكالة الأنباء السعودية (واس)

──────────────────────────────────